# New approach to decision trees for transition

In [1]:
using CovidSim_ilm

┌ Info: Precompiling CovidSim_ilm [top-level]
└ @ Base loading.jl:1317


In [2]:
using StatsBase
using DelimitedFiles
using Distributions
using PrettyPrint
using JSON
using YAML

In [3]:
cd(joinpath(homedir(),"Dropbox/Covid Modeling/Covid-ILM/source"))

## Use an input format closer to the intended output format

In [9]:
newdtfname = "../parameters/new.yml"

"../parameters/new.yml"

In [10]:
newdt = YAML.load_file(newdtfname)

Dict{Any, Any} with 5 entries:
  5 => Dict{Any, Any}(5=>Dict{Any, Any}(5=>Dict{Any, Any}("probs"=>[0.1, 0.5, 0…
  4 => Dict{Any, Any}(5=>Dict{Any, Any}(5=>Dict{Any, Any}("probs"=>[0.15, 0.6, …
  2 => Dict{Any, Any}(5=>Dict{Any, Any}(5=>Dict{Any, Any}("probs"=>[0.2, 0.7, 0…
  3 => Dict{Any, Any}(5=>Dict{Any, Any}(5=>Dict{Any, Any}("probs"=>[0.2, 0.7, 0…
  1 => Dict{Any, Any}(5=>Dict{Any, Any}(5=>Dict{Any, Any}("probs"=>[0.4, 0.5, 0…

In [11]:
newdt[1]

Dict{Any, Any} with 5 entries:
  5  => Dict{Any, Any}(5=>Dict{Any, Any}("probs"=>[0.4, 0.5, 0.1], "outcomes"=>…
  25 => Dict{Any, Any}(7=>Dict{Any, Any}("probs"=>[0.976, 0.024], "outcomes"=>[…
  9  => Dict{Any, Any}(5=>Dict{Any, Any}("probs"=>[0.9, 0.1], "outcomes"=>[3, 7…
  14 => Dict{Any, Any}(6=>Dict{Any, Any}("probs"=>[1.0], "outcomes"=>[3]), 7=>D…
  19 => Dict{Any, Any}(8=>Dict{Any, Any}("probs"=>[0.891, 0.106, 0.003], "outco…

In [12]:
newdt[1][5]

Dict{Any, Any} with 1 entry:
  5 => Dict{Any, Any}("probs"=>[0.4, 0.5, 0.1], "outcomes"=>[5, 6, 7])

In [13]:
trees = setup_dt(newdtfname)

Dict{agegrp, Dict{Int64, Dict{condition, Dict{Symbol, Vector{T} where T}}}} with 5 entries:
  age80_up => Dict(5=>Dict(nil=>Dict(:outcomes=>condition[nil, mild, sick], :pr…
  age60_79 => Dict(5=>Dict(nil=>Dict(:outcomes=>condition[nil, mild, sick], :pr…
  age0_19  => Dict(5=>Dict(nil=>Dict(:outcomes=>condition[nil, mild, sick], :pr…
  age20_39 => Dict(5=>Dict(nil=>Dict(:outcomes=>condition[nil, mild, sick], :pr…
  age40_59 => Dict(5=>Dict(nil=>Dict(:outcomes=>condition[nil, mild, sick], :pr…

In [14]:
display_tree(trees)

agegrp: age80_up =>
    duration: 5 =>
        fromcond: nil =>
            probs: => [0.1, 0.5, 0.4]
            outcomes: => condition[nil, mild, sick]
    duration: 25 =>
        fromcond: severe =>
            probs: => [0.676, 0.324]
            outcomes: => status[recovered, dead]
        fromcond: sick =>
            probs: => [0.682, 0.318]
            outcomes: => status[recovered, dead]
    duration: 9 =>
        fromcond: sick =>
            probs: => [0.6, 0.4]
            outcomes: => condition[sick, severe]
        fromcond: mild =>
            probs: => [0.4, 0.6]
            outcomes: => condition[mild, sick]
        fromcond: nil =>
            probs: => [0.5, 0.5]
            outcomes: => Enum{Int32}[recovered, sick]
    duration: 14 =>
        fromcond: severe =>
            probs: => [0.12, 0.67, 0.21]
            outcomes: => Enum{Int32}[recovered, severe, dead]
        fromcond: sick =>
            probs: => [0.7, 0.1, 0.2]
            outcomes: => Enum{Int32}[rec

In [15]:
trees[age80_up]

Dict{Int64, Dict{condition, Dict{Symbol, Vector{T} where T}}} with 5 entries:
  5  => Dict(nil=>Dict(:outcomes=>condition[nil, mild, sick], :probs=>[0.1, 0.5…
  25 => Dict(severe=>Dict(:outcomes=>status[recovered, dead], :probs=>[0.676, 0…
  9  => Dict(sick=>Dict(:outcomes=>condition[sick, severe], :probs=>[0.6, 0.4])…
  14 => Dict(severe=>Dict(:outcomes=>Enum{Int32}[recovered, severe, dead], :pro…
  19 => Dict(severe=>Dict(:outcomes=>Enum{Int32}[recovered, severe, dead], :pro…

In [16]:
trees[age80_up][5]

Dict{condition, Dict{Symbol, Vector{T} where T}} with 1 entry:
  nil => Dict(:outcomes=>condition[nil, mild, sick], :probs=>[0.1, 0.5, 0.4])

In [17]:
trees[age80_up][5][nil]

Dict{Symbol, Vector{T} where T} with 2 entries:
  :outcomes => condition[nil, mild, sick]
  :probs    => [0.1, 0.5, 0.4]

#### does it check out?

In [18]:
seqs = CovidSim_ilm.walksequence(trees[age80_up])

31-element Vector{Any}:
 NamedTuple{(:duration, :fromcond, :tocond, :prob), T} where T<:Tuple[(duration = 5, fromcond = nil, tocond = nil, prob = 0.1), (duration = 9, fromcond = nil, tocond = recovered, prob = 0.5)]
 NamedTuple{(:duration, :fromcond, :tocond, :prob), T} where T<:Tuple[(duration = 5, fromcond = nil, tocond = nil, prob = 0.1), (duration = 9, fromcond = nil, tocond = sick, prob = 0.5), (duration = 14, fromcond = sick, tocond = recovered, prob = 0.7)]
 NamedTuple{(:duration, :fromcond, :tocond, :prob), T} where T<:Tuple[(duration = 5, fromcond = nil, tocond = mild, prob = 0.5), (duration = 9, fromcond = mild, tocond = mild, prob = 0.4), (duration = 14, fromcond = mild, tocond = recovered, prob = 0.7)]
 NamedTuple{(:duration, :fromcond, :tocond, :prob), T} where T<:Tuple[(duration = 5, fromcond = nil, tocond = mild, prob = 0.5), (duration = 9, fromcond = mild, tocond = sick, prob = 0.6), (duration = 14, fromcond = sick, tocond = recovered, prob = 0.7)]
 NamedTuple{(:duratio

In [19]:
CovidSim_ilm.verifyprobs(seqs)

(Dict{status, Float64}(dead => 0.14975755200000002, recovered => 0.8502424480000002), 1.0000000000000002)

## Test and Re-write sanity check for transition phases

In [36]:
CovidSim_ilm.sanitycheck(trees)

for agegroup age0_19
    dead => 0.0004885224
    recovered => 0.9995114776000001
    Prob total: 1.0
for agegroup age20_39
    dead => 0.0006683356800000001
    recovered => 0.9993316643200001
    Prob total: 1.0
for agegroup age40_59
    dead => 0.0038404027200000003
    recovered => 0.9961595972799999
    Prob total: 1.0
for agegroup age60_79
    dead => 0.019136634000000007
    recovered => 0.9808633660000001
    Prob total: 1.0
for agegroup age80_up
    dead => 0.14975755200000002
    recovered => 0.8502424480000002
    Prob total: 1.0000000000000002
